In [0]:
import pyspark.sql.functions as F
import pyspark.sql.types as T
from pyspark.sql.functions import col

In [0]:
df = spark.read.table('olist_dataset.bronze.items')

display(df.limit(10))

In [0]:
date_trimed_df = df.withColumn('shipping_date',col('shipping_date').cast(T.DateType()))

In [0]:
r_df = spark.read.table('olist_dataset.bronze.reviews')

display(r_df)

In [0]:
trimmied_r_df = r_df.select('order_id','review_score')

In [0]:
joined_df = date_trimed_df.join(trimmied_r_df, on= 'order_id', how= 'left').select('order_id','product_id','seller_id','item_id','shipping_date','price','shipping_cost',trimmied_r_df['review_score'])

In [0]:
null_values = joined_df.select([F.count(F.when(col(c).isNull(), c)).alias(c) for c in joined_df.columns])
null_values.show()

In [0]:
no_null_df = joined_df.withColumn('has_reviewed',col("review_score").isNotNull())\
    .fillna({'review_score':0})

display(no_null_df)

In [0]:
no_null_df.write.format('delta')\
    .option('mergeSchema',True)\
    .mode('overwrite')\
    .saveAsTable('olist_dataset.silver.items')